# Notebook 2 — Foreign-key mapping & relational stitching (Person B)

Takes Person A's harvested tables (`pmos_knowledge_base.db`, `data/clean/*.csv`) and:

1. Creates FK-constrained link tables (gene↔protein, protein↔pathway, entity↔publication, gene↔PCOS association, biomarker/hormone↔protein, therapeutic↔gene candidates).
2. Provides multi-table SQL search.
3. Assembles everything into the `Entity`/`Relationship` shapes Person C's frontend (`pmos-biograph`) expects.

All of the logic lives in `backend/pmos_data.py` (importable, and reused by `backend/server.py`, the API that actually serves the frontend). This notebook is a walkthrough of what that module does and why.

In [ ]:
import sys
sys.path.insert(0, 'backend')
import pmos_data as pd

conn = pd.get_connection()
pd.init_relationships(conn)  # idempotent: safe to re-run after Person A re-harvests data
print('relationship tables ready')

## 1. What got linked

Row counts for each FK-stitched table, straight from SQL.

In [ ]:
for table in [
    'gene_protein_link', 'protein_pathway_link', 'entity_publication_link',
    'gene_pcos_association', 'biomarker_protein_link', 'hormone_protein_link',
    'therapeutic_target_candidate',
]:
    n = conn.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table:32s} {n}')

## 2. Multi-table SQL search

`sql_search` is a literal `UNION ALL` + `LIKE` query across all seven entity tables — no Python ranking, just SQL.

In [ ]:
for row in pd.sql_search(conn, 'insulin', limit=10):
    print(row['entity_type'], '|', row['name'], '|', row['primary_id'])

## 3. Gene → protein → pathway, end to end

`build_index` assembles every table into the frontend's `Entity`/`Relationship` shapes and an adjacency map. This is the same object `backend/server.py` builds once at startup to answer every API request.

In [ ]:
index = pd.build_index(conn)
print('entities:', len(index.entities), '| relationships:', len(index.relationships))

for n in pd.get_neighbours(index, 'gene-cyp17a1'):
    print(n['direction'], n['relationship']['label'], '->', n['entity']['id'])

## 4. Dataset summary (same payload the frontend's Overview page renders)

In [ ]:
import json
print(json.dumps(pd.get_summary(index), indent=2))

## 5. Serving this to Person C's frontend

`backend/server.py` wraps the functions above in a zero-dependency `http.server` that implements the exact REST contract `pmos-biograph/pmos-biograph/src/services/api.config.ts` expects. Run it with `py backend/server.py`, then set `VITE_USE_MOCK=false` in the frontend's `.env` — see `backend/README.md` for the full handoff notes, including the one caveat worth remembering: `therapeutic_target_candidate` rows are text-matched, not curated, and are surfaced to the UI as `demo-unvalidated` so they render with a visible unverified badge rather than being presented as fact.